# Exposição do emprego formal à IA generativa nos estados brasileiros

Este caderno permite explorar os dados do trabalho **Quais estados e regiões do Brasil concentram mais empregos formais expostos à IA generativa?** (Metodologia e Técnicas de Pesquisa em Economia II, UFMT, 2026).

**Como usar:** clique em *Ambiente de execução* > *Executar tudo* (ou execute as células na ordem, com Shift+Enter). Depois, use os menus para escolher estado, setor ou indicador. Nada precisa ser instalado.

**Definições rápidas**
- A OIT dá a cada profissão uma nota de 1 a 4, conforme a parte das tarefas que a IA generativa poderia executar.
- **Exposta** = nota de 1 a 4. **Alta exposição** = nota 3 ou 4.
- Os empregos vêm da RAIS 2025 (vínculos formais ativos em 31/12/2025).
- A nota mede potencial técnico, não perda de emprego. A informalidade fica de fora.

## 1. Carregar os dados

In [ ]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display

URL = "https://github.com/Gleidisonjr/ia-generativa-emprego-formal-brasil/raw/main/base_dados.xlsx"

# Lê todas as abas da planilha de uma vez
dados = pd.read_excel(URL, sheet_name=None, dtype={"cbo": str, "codigo_ibge": str, "isco08": str})
print("Abas carregadas:", ", ".join(dados.keys()))

estados  = dados["Estados"]
regioes  = dados["Regioes"]
setores  = dados["Setores_por_estado"]
decomp   = dados["Decomposicao"]
profs    = dados["Profissoes_por_estado"]
munic    = dados["Municipios"]
salario  = dados["Remuneracao_estados"]

def vir(x, sinal=False):
    """Formata número com uma casa decimal e vírgula."""
    return (f"{x:+.1f}" if sinal else f"{x:.1f}").replace(".", ",")

AZUL, CLARO, LARANJA = "#1f4e79", "#a9c6e3", "#d98c2b"
plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False, "figure.dpi": 110})

## 2. Visão geral do Brasil

In [ ]:
br = regioes[regioes.regiao == "Brasil"].iloc[0]
print(f"Vínculos analisados: {br.vinculos_analisados:,.0f}".replace(",", "."))
print("Em ocupações expostas: " + vir(br.pct_expostos) + "%")
print("Em alta exposição: " + vir(br.pct_alta_exposicao) + "%")
print()
display(regioes)

## 3. Qual estado está mais exposto?
Escolha o indicador e, se quiser, filtre por região.

In [ ]:
def ranking(indicador="pct_expostos", regiao="Todas"):
    d = estados if regiao == "Todas" else estados[estados.regiao == regiao]
    d = d.sort_values(indicador)
    media = br.pct_expostos if indicador == "pct_expostos" else br.pct_alta_exposicao
    fig, ax = plt.subplots(figsize=(7, 0.28 * len(d) + 1.5))
    ax.barh(d.uf, d[indicador], color=AZUL)
    ax.axvline(media, color=LARANJA, ls="--", lw=1.2)
    ax.text(media, len(d) - 0.3, f"  Brasil: {media:.1f}%", color=LARANJA, va="bottom", fontsize=9)
    nome = "Empregos em ocupações expostas (%)" if indicador == "pct_expostos" else "Empregos em alta exposição (%)"
    ax.set_xlabel(nome)
    ax.set_title(f"{nome}, por estado")
    plt.tight_layout(); plt.show()

widgets.interact(ranking,
    indicador=widgets.Dropdown(options=[("Expostos (nota 1 a 4)", "pct_expostos"), ("Alta exposição (nota 3 ou 4)", "pct_alta_exposicao")], description="Indicador"),
    regiao=widgets.Dropdown(options=["Todas", "Norte", "Nordeste", "Sudeste", "Sul", "Centro-Oeste"], description="Região"));

## 4. Olhe um estado de perto
Escolha um estado para ver os números, os setores que mais empregam pessoas em ocupações expostas e as profissões mais numerosas entre elas.

In [ ]:
def estado(uf="MT"):
    e = estados[estados.uf == uf].iloc[0]
    print(f"{e.estado} ({e.regiao})  |  posição {e.posicao} de 27")
    print("Em ocupações expostas: " + vir(e.pct_expostos) + "%  |  em alta exposição: " + vir(e.pct_alta_exposicao) + "%")
    dc = decomp[decomp.uf == uf].iloc[0]
    print("Diferença para o Brasil: " + vir(dc.diferenca_vs_brasil_pp, True) + " p.p. (setor: " + vir(dc.efeito_setor_pp, True) + "; profissão dentro do setor: " + vir(dc.efeito_profissao_no_setor_pp, True) + ")")
    s = setores[setores.uf == uf].sort_values("vinculos_expostos", ascending=False).head(8)
    fig, ax = plt.subplots(figsize=(8, 3.6))
    ax.barh(s.setor.str.slice(0, 48)[::-1], s.vinculos_expostos[::-1], color=AZUL)
    ax.set_xlabel("Vínculos em ocupações expostas"); ax.set_title(f"{e.estado}: setores com mais empregos expostos")
    plt.tight_layout(); plt.show()
    p = profs[(profs.uf == uf) & (profs.exposta_gradiente_1a4 > 0.5)].sort_values("vinculos", ascending=False).head(10)
    display(p[["profissao", "vinculos", "remuneracao_media", "escore_oit"]].rename(columns={"profissao": "Profissão", "vinculos": "Vínculos", "remuneracao_media": "Remuneração média (R$)", "escore_oit": "Nota OIT"}).reset_index(drop=True))

widgets.interact(estado, uf=widgets.Dropdown(options=list(estados.sort_values("uf").uf), value="MT", description="Estado"));

## 5. Compare dois estados

In [ ]:
def comparar(a="DF", b="MS"):
    cols = ["estado", "regiao", "pct_expostos", "pct_alta_exposicao", "vinculos_analisados"]
    display(estados[estados.uf.isin([a, b])][cols].set_index("estado").T)
    d = decomp[decomp.uf.isin([a, b])].set_index("uf")[["diferenca_vs_brasil_pp", "efeito_setor_pp", "efeito_profissao_no_setor_pp"]]
    d.columns = ["Diferença total (p.p.)", "Efeito do setor", "Efeito da profissão"]
    ax = d.T.plot.bar(figsize=(6.5, 3.2), color=[AZUL, LARANJA], rot=0)
    ax.axhline(0, color="black", lw=0.8); ax.set_title("Por que cada estado difere do Brasil?"); ax.set_ylabel("pontos percentuais")
    plt.tight_layout(); plt.show()

ufs = list(estados.sort_values("uf").uf)
widgets.interact(comparar, a=widgets.Dropdown(options=ufs, value="DF", description="Estado A"), b=widgets.Dropdown(options=ufs, value="MS", description="Estado B"));

## 6. Setor por setor
Escolha um setor da economia e veja em quais estados ele tem mais empregos em ocupações expostas (em % dos vínculos do próprio setor).

In [ ]:
def por_setor(setor):
    d = setores[(setores.setor == setor) & (setores.vinculos_analisados >= 500)].sort_values("pct_expostos")
    fig, ax = plt.subplots(figsize=(7, 0.28 * len(d) + 1.5))
    ax.barh(d.uf, d.pct_expostos, color=AZUL)
    ax.set_xlabel("Vínculos do setor em ocupações expostas (%)"); ax.set_title(setor[:70])
    plt.tight_layout(); plt.show()

ordem = setores.groupby("setor").vinculos_analisados.sum().sort_values(ascending=False).index.tolist()
widgets.interact(por_setor, setor=widgets.Dropdown(options=ordem, description="Setor", layout=widgets.Layout(width="650px")));

## 7. Exposição e remuneração
Os estados mais expostos pagam mais? Cada ponto é um estado.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.5))
ax.scatter(salario.remuneracao_media, salario.pct_expostos, color=AZUL, s=40)
for _, r in salario.iterrows():
    ax.annotate(r.uf, (r.remuneracao_media, r.pct_expostos), xytext=(4, 3), textcoords="offset points", fontsize=8)
ax.set_xlabel("Remuneração média mensal (R$)"); ax.set_ylabel("Empregos em ocupações expostas (%)")
ax.set_title("Exposição e remuneração média, por estado")
plt.tight_layout(); plt.show()
print("Correlação entre as duas medidas: %.2f" % salario.remuneracao_media.corr(salario.pct_expostos))
display(dados["Remuneracao_decis"])

## 8. Municípios
Escolha um estado e veja os municípios mais e menos expostos (apenas os que têm ao menos o número mínimo de vínculos, para evitar números instáveis).

In [ ]:
def municipios(uf="MT", minimo=1000):
    d = munic[(munic.uf == uf) & (munic.vinculos_analisados >= minimo)].sort_values("pct_expostos", ascending=False)
    if d.empty:
        print("Nenhum município com esse mínimo de vínculos."); return
    cols = ["municipio", "vinculos_analisados", "pct_expostos", "pct_alta_exposicao"]
    print("Mais expostos"); display(d.head(10)[cols].reset_index(drop=True))
    print("Menos expostos"); display(d.tail(10)[cols].reset_index(drop=True))

widgets.interact(municipios, uf=widgets.Dropdown(options=ufs, value="MT", description="Estado"), minimo=widgets.IntSlider(value=1000, min=0, max=20000, step=500, description="Mín. vínculos"));

## 9. Faça a sua própria pergunta
As tabelas estão carregadas como `estados`, `regioes`, `setores`, `decomp`, `profs`, `munic` e `salario`. Edite a célula abaixo. Exemplo: quais são as 10 profissões com mais empregos em alta exposição no Brasil?

In [ ]:
top = (profs[profs.alta_exposicao_gradiente_3a4 > 0.5]
       .groupby("profissao").vinculos.sum()
       .sort_values(ascending=False).head(10))
top

## Sobre os dados

- **Base completa em Excel:** `base_dados.xlsx` (uma aba por tabela, com filtros). A aba *LEIA-ME* explica cada uma.
- **Fontes:** Gmyrek et al. (2025), ILO Working Paper 140 (notas por ocupação); RAIS 2025 via Base dos Dados (empregos); tabela de correspondência CBO-2002 e ISCO-08 do Ministério do Trabalho (pacote `ocupacoesBR`).
- Cerca de 7% dos vínculos não tiveram correspondência entre as classificações; os percentuais valem para os 93% restantes.